# ARKENSTONE MASTER GPU v3 — Causal Retention → Transfer → Recovery

**Branch:** `Arkenstone`  
**Binding campaign plan:** `3d98103cacc38177390df78ee0eff402da687fcf`

Run this notebook top-to-bottom on **Colab T4 GPU** (`Runtime → Change runtime type → T4 GPU`).

This notebook fixes the previous `xla_sync()` failure and runs the current highest-value experiments in one session:

1. **ARK-007R — fresh-checkpoint replication**  
   3 fresh arithmetic acquisition seeds × 4 frozen continuation orders, paired `LR=1e-3` vs `LR=1e-5`.

2. **ARK-009 — non-arithmetic transfer**  
   A compact symbolic variable-binding/retrieval task. First prove the Micro model can acquire it; only then run matched retention forks.

3. **ARK-010 — recovery after collapse**  
   When a high-LR continuation actually collapses, fork the exact collapsed state and test whether `LR=1e-5` can recover capability versus continued high LR.

The notebook is budget-aware, saves partial receipts, and downloads a ZIP of all JSON results at the end.

**Claim discipline:** a completed run can establish micro-task causal effects. It cannot establish a universal optimizer law or an AGI mechanism.


In [ ]:
# ===== ARKENSTONE MASTER GPU v3: SHARED HARNESS =====
import os, json, math, time, hashlib, random, copy, zipfile
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

PLAN_SHA = "3d98103cacc38177390df78ee0eff402da687fcf"
CANONICAL_T2_SHA = "0dd9305697045b0fbf4e7f268b46a4d7276e4794af5d78b60e999df914ae4236"
RESULTS_DIR = Path("/content/arkenstone_v3_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BUDGET_MINUTES = 240  # edit if your Colab allocation differs
SESSION_START = time.time()
def minutes_left():
    return BUDGET_MINUTES - (time.time() - SESSION_START) / 60.0

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. In Colab choose Runtime -> Change runtime type -> T4 GPU, then Run all."
    )
DEVICE = torch.device("cuda")
print("DEVICE:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

def device_sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

# Backward-compatible fix for the previous notebook's NameError.
# XLA needs an explicit mark_step; CUDA does not. Keep this a no-op for speed.
def xla_sync():
    return None

def sha_json(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str).encode("utf-8")
    ).hexdigest()

def cpu_tree(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu().clone()
    if isinstance(obj, dict):
        return {k: cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [cpu_tree(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(cpu_tree(v) for v in obj)
    return copy.deepcopy(obj)

def save_result(name, payload):
    payload = dict(payload)
    payload.setdefault("plan_commit_sha", PLAN_SHA)
    payload.setdefault("device", str(DEVICE))
    payload.setdefault("torch", torch.__version__)
    body = dict(payload)
    body.pop("receipt_sha256", None)
    payload["receipt_sha256"] = sha_json(body)
    path = RESULTS_DIR / name
    path.write_text(json.dumps(payload, indent=2, default=str) + "\n", encoding="utf-8")
    print("saved:", path)
    return path

# ---------- Canonical ARK-002B T2 manifest ----------
def _rows(split, n, ds=13):
    rng = random.Random(ds)
    tens = range(1, 6) if split == "train" else range(6, 8)
    rows, seen, guard = [], set(), 0
    while len(rows) < n and guard < 2_000_000:
        guard += 1
        ta = rng.choice(list(tens)); ua = rng.randrange(10)
        tb = rng.randrange(1, 10 - ta); ub = rng.randrange(0, 10 - ua)
        a, b = ta * 10 + ua, tb * 10 + ub
        if (a, b) in seen:
            continue
        seen.add((a, b))
        rows.append((f"{a} + {b} = ", f"{a+b}"))
    assert len(rows) == n
    return rows

def build_t2_manifest():
    train = _rows("train", 500)
    raw_test = _rows("test", 260)
    train_pairs = {
        tuple(sorted((int(p.split("+")[0]), int(p.split("+")[1].split("=")[0]))))
        for p, _ in train
    }
    test = []
    for p, a in raw_test:
        pair = tuple(sorted((int(p.split("+")[0]), int(p.split("+")[1].split("=")[0]))))
        if pair in train_pairs:
            continue
        test.append((p, a))
        if len(test) == 200:
            break
    manifest = {"train": train, "test": test}
    manifest["sha"] = hashlib.sha256(
        json.dumps({"train": train, "test": test}, sort_keys=True).encode()
    ).hexdigest()
    assert manifest["sha"] == CANONICAL_T2_SHA, (manifest["sha"], CANONICAL_T2_SHA)
    return manifest

T2 = build_t2_manifest()
print("T2 manifest:", T2["sha"], "|", len(T2["train"]), "train,", len(T2["test"]), "test")

# ---------- Exact historical ARK-001 Micro semantics ----------
COMPACT = ["<pad>", "<bos>", "<eos>", "0","1","2","3","4","5","6","7","8","9","+","-","*","/","="," "]

class CompactVocab:
    PAD, BOS, EOS = 0, 1, 2
    def __init__(self):
        self.table = {t: i for i, t in enumerate(COMPACT)}
        self.size = len(COMPACT)
    def extend(self, text):
        for ch in text:
            if ch not in self.table:
                self.table[ch] = self.size
                self.size += 1
    def encode(self, text):
        return [self.BOS] + [self.table[ch] for ch in text]
    def decode(self, ids):
        inv = {i: t for t, i in self.table.items()}
        return "".join(inv.get(i, "") for i in ids if i not in (self.PAD, self.BOS, self.EOS))

class RMSNorm(nn.Module):
    def __init__(self, width, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps
    def forward(self, x):
        # Historical ARK-001 behavior: the weight parameter exists but is not multiplied.
        return x * torch.rsqrt(
            x.float().square().mean(-1, keepdim=True) + self.eps
        ).to(x.dtype)

class Block(nn.Module):
    def __init__(self, width, heads=4, ffn=512):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = RMSNorm(width), RMSNorm(width)
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.proj = nn.Linear(width, width, bias=False)
        self.gate = nn.Linear(width, ffn, bias=False)
        self.up = nn.Linear(width, ffn, bias=False)
        self.down = nn.Linear(ffn, width, bias=False)
    def forward(self, x):
        b, t, w = x.shape
        h = self.norm1(x)
        q, k, v = self.qkv(h).chunk(3, dim=-1)
        hd = w // self.heads
        q = q.view(b, t, self.heads, hd).transpose(1, 2)
        k = k.view(b, t, self.heads, hd).transpose(1, 2)
        v = v.view(b, t, self.heads, hd).transpose(1, 2)
        pos = torch.arange(t, device=x.device)
        inv = 10000.0 ** (-torch.arange(0, hd, 2, device=x.device).float() / hd)
        phase = pos.float()[:, None] * inv[None, :]
        cos, sin = phase.cos()[None, None], phase.sin()[None, None]
        def apply_rope(z):
            ze, zo = z[..., 0::2], z[..., 1::2]
            cos_e = cos.squeeze(0).squeeze(0)
            sin_e = sin.squeeze(0).squeeze(0)
            cos_f = torch.repeat_interleave(cos_e, 2, dim=-1)[None, None]
            sin_f = torch.repeat_interleave(sin_e, 2, dim=-1)[None, None]
            return torch.cat(
                (
                    ze * cos_f[..., 0::2] - zo * sin_f[..., 0::2],
                    ze * sin_f[..., 0::2] + zo * cos_f[..., 0::2],
                ),
                dim=-1,
            )
        q, k = apply_rope(q), apply_rope(k)
        mask = torch.tril(torch.ones(t, t, dtype=torch.bool, device=x.device))
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        out = out.transpose(1, 2).contiguous().view(b, t, w)
        x = x + self.proj(out)
        h = self.norm2(x)
        return x + self.down(F.silu(self.gate(h)) * self.up(h))

class Micro(nn.Module):
    def __init__(self, vocab_size, width=128, layers=4, ffn=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, width)
        self.blocks = nn.ModuleList(Block(width, 4, ffn) for _ in range(layers))
        self.norm = RMSNorm(width)
        self.width = width
    def forward(self, ids):
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x)
        return self.norm(x) @ self.embed.weight.T

def encode_batch(vocab, rows, device):
    prompts = [vocab.encode(p) for p, _ in rows]
    answers = [vocab.encode(a) + [vocab.EOS] for _, a in rows]
    length = max(len(p) + len(a) for p, a in zip(prompts, answers))
    tokens = torch.full((len(rows), length), vocab.PAD, dtype=torch.long)
    prompt_len = torch.zeros(len(rows), dtype=torch.long)
    for i, (p, a) in enumerate(zip(prompts, answers)):
        tokens[i, :len(p)] = torch.tensor(p)
        tokens[i, len(p):len(p)+len(a)] = torch.tensor(a)
        prompt_len[i] = len(p)
    return tokens.to(device), prompt_len.to(device)

def loss_fn(model, vocab, rows, device):
    tokens, prompt_len = encode_batch(vocab, rows, device)
    logits = model(tokens[:, :-1])
    targets = tokens[:, 1:]
    positions = torch.arange(tokens.shape[1] - 1, device=device)[None, :]
    mask = positions >= (prompt_len - 1)[:, None]  # exact historical ARK-001 rule
    if mask.sum() == 0:
        raise ValueError("no supervised targets")
    losses = F.cross_entropy(
        logits.float().reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        reduction="none",
    ).view(targets.shape)
    count = int(mask.sum().item())
    return (losses * mask).sum() / count, count

@torch.no_grad()
def greedy_exact(model, vocab, rows, device, max_answer=8):
    model.eval()
    groups = {}
    for index, (prompt, _) in enumerate(rows):
        groups.setdefault(len(vocab.encode(prompt)), []).append(index)
    text_of = {}
    for same_length in groups.values():
        batch_rows = [rows[i] for i in same_length]
        tokens = torch.tensor([vocab.encode(p) for p, _ in batch_rows], device=device)
        batch = len(batch_rows)
        finished = torch.zeros(batch, dtype=torch.bool)
        generated = [[] for _ in batch_rows]
        for _ in range(max_answer):
            logits = model(tokens)[:, -1]
            next_ids = torch.argmax(logits, dim=-1)
            tokens = torch.cat(
                [tokens, torch.full((batch, 1), vocab.PAD, dtype=torch.long, device=device)],
                dim=1,
            )
            all_finished = True
            for i in range(batch):
                if finished[i]:
                    continue
                token = int(next_ids[i].item())
                if token in (vocab.EOS, vocab.PAD):
                    finished[i] = True
                else:
                    generated[i].append(token)
                    tokens[i, -1] = token
                    all_finished = False
            if all_finished:
                break
        for i, (_prompt, _answer) in enumerate(batch_rows):
            text_of[same_length[i]] = vocab.decode(generated[i]).strip()
    correct = sum(1 for index, (_, answer) in enumerate(rows) if text_of[index] == answer)
    model.train()
    return correct / max(len(rows), 1)

def detect_sustained(eval_steps, values, bar=0.90, consec=3, below=False):
    streak, onset = 0, None
    for step, value in zip(eval_steps, values):
        hit = value < bar if below else value >= bar
        if hit:
            if streak == 0:
                onset = step
            streak += 1
            if streak >= consec:
                return onset, step
        else:
            streak, onset = 0, None
    return None, None

def flat_params(model):
    return torch.cat([p.detach().reshape(-1) for p in model.parameters()])

@torch.no_grad()
def param_displacement(model, reference):
    cur = flat_params(model)
    ref = reference.to(cur.device)
    d = (cur - ref).norm().item()
    b = ref.norm().item()
    return {"l2_distance": d, "relative_displacement": d / max(b, 1e-12)}

def generate_indices(seed, n_steps, batch_size, pool_size):
    g = torch.Generator().manual_seed(seed)
    return [
        torch.randint(0, pool_size, (batch_size,), generator=g).tolist()
        for _ in range(n_steps)
    ]

def order_hash(indices):
    return sha_json(indices)

def make_optimizer(model, lr):
    return torch.optim.AdamW(
        model.parameters(), lr=lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1
    )

def snapshot_training_state(model, optimizer):
    return {
        "model": cpu_tree(model.state_dict()),
        "optimizer": cpu_tree(optimizer.state_dict()),
        "torch_rng": torch.get_rng_state().cpu().clone(),
        "cuda_rng": [x.cpu().clone() for x in torch.cuda.get_rng_state_all()],
    }

def restore_training_state(vocab_size, snapshot, lr):
    model = Micro(vocab_size, 128).to(DEVICE)
    model.load_state_dict({k: v.to(DEVICE) for k, v in snapshot["model"].items()})
    opt = make_optimizer(model, lr)
    opt.load_state_dict(snapshot["optimizer"])
    for group in opt.param_groups:
        group["lr"] = lr
    torch.set_rng_state(snapshot["torch_rng"])
    torch.cuda.set_rng_state_all(snapshot["cuda_rng"])
    return model, opt

def retention_metrics(trajectory):
    if not trajectory:
        return {"status": "EMPTY"}
    steps = [e["step"] for e in trajectory]
    vals = [e["test_exact"] for e in trajectory]
    onset, confirm = detect_sustained(steps, vals, bar=0.90, consec=3, below=True)
    return {
        "RET90": sum(v >= 0.90 for v in vals) / len(vals),
        "RET50": sum(v >= 0.50 for v in vals) / len(vals),
        "OOD_AREA": sum(vals) / len(vals),
        "T_COLLAPSE_90_ONSET": onset,
        "T_COLLAPSE_90_CONFIRM": confirm,
        "FINAL_OOD": vals[-1],
        "PEAK_OOD": max(vals),
        "STABILITY_GAP": max(vals) - vals[-1],
        "collapse_binary": confirm is not None,
    }

print("Harness ready. Historical Micro/loss semantics preserved; xla_sync() is CUDA-safe.")


## ARK-007R — fresh independent checkpoint replication
Runs first because it directly strengthens or weakens the current strongest causal result.


In [ ]:
# ===== ARK-007R: FRESH-CHECKPOINT RETENTION REPLICATION =====
ARK007R_ACQ_SEEDS = [909, 1010, 1111]
ARK007R_CONT_SEEDS = [2701, 2702, 2703, 2704]
ARK007R_POST_STEPS = 6000
ARK010_RECOVERY_STEPS = 4000
BATCH = 64
EVAL_EVERY = 200

ark007r_checkpoints = {}
ark007r_results = []
ark010_collapse_snapshots = []

def acquire_t2(seed):
    torch.manual_seed(seed)
    vocab = CompactVocab()
    model = Micro(vocab.size, 128).to(DEVICE)
    opt = make_optimizer(model, 1e-3)
    rng = torch.Generator().manual_seed(seed)
    eval_steps, eval_ood, traj = [], [], []
    tokens = 0
    t0 = time.time()
    onset = confirm = None
    for step in range(1, 28001):
        idx = torch.randint(0, len(T2["train"]), (BATCH,), generator=rng)
        rows = [T2["train"][i] for i in idx]
        loss, sup = loss_fn(model, vocab, rows, DEVICE)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        xla_sync()
        tokens += sup
        if step % EVAL_EVERY == 0 or step == 1:
            tr = greedy_exact(model, vocab, T2["train"][:100], DEVICE)
            te = greedy_exact(model, vocab, T2["test"], DEVICE)
            traj.append({"step": step, "train_exact": tr, "test_exact": te})
            eval_steps.append(step); eval_ood.append(te)
            onset, confirm = detect_sustained(eval_steps, eval_ood, 0.90, 3, below=False)
            if confirm is not None:
                snap = snapshot_training_state(model, opt)
                return {
                    "status": "ACQUIRED",
                    "seed": seed,
                    "vocab": vocab,
                    "snapshot": snap,
                    "reference_flat": flat_params(model).detach().cpu(),
                    "g90_onset_step": onset,
                    "g90_confirmation_step": confirm,
                    "first_treated_optimizer_step": confirm + 1,
                    "acquisition_steps": step,
                    "acquisition_supervised_tokens": tokens,
                    "trajectory": traj,
                }
        if time.time() - t0 > 1500 or minutes_left() < 20:
            break
    return {
        "status": "NO_G90",
        "seed": seed,
        "g90_onset_step": onset,
        "g90_confirmation_step": confirm,
        "acquisition_steps": step,
        "acquisition_supervised_tokens": tokens,
        "trajectory": traj,
    }

def run_t2_continuation(acq, continuation_seed, arm_name, arm_lr):
    full_indices = generate_indices(
        continuation_seed,
        ARK007R_POST_STEPS + ARK010_RECOVERY_STEPS,
        BATCH,
        len(T2["train"]),
    )
    prefix = full_indices[:ARK007R_POST_STEPS]
    model, opt = restore_training_state(acq["vocab"].size, acq["snapshot"], arm_lr)
    traj, tokens = [], 0
    low_streak, collapse_onset = 0, None
    collapse_snapshot = None

    for treated_step, batch_idx in enumerate(prefix, start=1):
        rows = [T2["train"][i] for i in batch_idx]
        loss, sup = loss_fn(model, acq["vocab"], rows, DEVICE)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        xla_sync()
        tokens += sup

        if treated_step % EVAL_EVERY == 0:
            te = greedy_exact(model, acq["vocab"], T2["test"], DEVICE)
            disp = param_displacement(model, acq["reference_flat"])
            traj.append({
                "step": treated_step,
                "test_exact": te,
                "supervised_tokens": tokens,
                **disp,
            })

            if te < 0.90:
                if low_streak == 0:
                    collapse_onset = treated_step
                low_streak += 1
                if low_streak >= 3 and collapse_snapshot is None and arm_name == "HIGH":
                    collapse_snapshot = {
                        "acquisition_seed": acq["seed"],
                        "continuation_seed": continuation_seed,
                        "collapse_onset_step": collapse_onset,
                        "collapse_confirmation_step": treated_step,
                        "snapshot": snapshot_training_state(model, opt),
                        "reference_flat": acq["reference_flat"],
                        "vocab_size": acq["vocab"].size,
                        "tail_indices": full_indices[treated_step:treated_step + ARK010_RECOVERY_STEPS],
                    }
            else:
                low_streak, collapse_onset = 0, None

    return {
        "acquisition_seed": acq["seed"],
        "continuation_seed": continuation_seed,
        "continuation_order_sha256": order_hash(full_indices),
        "arm": arm_name,
        "lr": arm_lr,
        "g90_onset_step": acq["g90_onset_step"],
        "g90_confirmation_step": acq["g90_confirmation_step"],
        "first_treated_optimizer_step": acq["first_treated_optimizer_step"],
        "post_confirmation_steps": ARK007R_POST_STEPS,
        "post_confirmation_supervised_tokens": tokens,
        "retention": retention_metrics(traj),
        "trajectory": traj,
        "collapse_snapshot": collapse_snapshot,
    }

for seed in ARK007R_ACQ_SEEDS:
    if minutes_left() < 35:
        print("ARK-007R budget gate before acquisition seed", seed)
        break
    print("\n=== ARK-007R acquire seed", seed, "===")
    acq = acquire_t2(seed)
    ark007r_checkpoints[seed] = acq
    print("status:", acq["status"], "onset:", acq.get("g90_onset_step"), "confirm:", acq.get("g90_confirmation_step"))
    if acq["status"] != "ACQUIRED":
        continue

    for cs in ARK007R_CONT_SEEDS:
        if minutes_left() < 18:
            print("ARK-007R budget gate before continuation", cs)
            break
        for arm_name, arm_lr in (("HIGH", 1e-3), ("LOW", 1e-5)):
            print(f"  seed {seed} order {cs} {arm_name}")
            result = run_t2_continuation(acq, cs, arm_name, arm_lr)
            if result["collapse_snapshot"] is not None:
                ark010_collapse_snapshots.append(result["collapse_snapshot"])
            # Don't serialize large snapshot tensors into the ARK-007R receipt.
            result["collapse_snapshot"] = (
                None if result["collapse_snapshot"] is None else {
                    k: v for k, v in result["collapse_snapshot"].items()
                    if k not in ("snapshot", "reference_flat", "tail_indices")
                }
            )
            ark007r_results.append(result)
            print("   ", result["retention"])

# Paired summary
pairs = {}
for r in ark007r_results:
    pairs.setdefault((r["acquisition_seed"], r["continuation_seed"]), {})[r["arm"]] = r

discordant_protect = discordant_reverse = both_collapse = both_stable = 0
high_c = low_c = 0
for key, p in pairs.items():
    if "HIGH" not in p or "LOW" not in p:
        continue
    hc = bool(p["HIGH"]["retention"]["collapse_binary"])
    lc = bool(p["LOW"]["retention"]["collapse_binary"])
    high_c += int(hc); low_c += int(lc)
    if hc and not lc: discordant_protect += 1
    elif lc and not hc: discordant_reverse += 1
    elif hc and lc: both_collapse += 1
    else: both_stable += 1

n_pairs = sum(1 for p in pairs.values() if "HIGH" in p and "LOW" in p)
risk_diff = (low_c / n_pairs - high_c / n_pairs) if n_pairs else None
fresh_seeds_completed = sorted({
    r["acquisition_seed"] for r in ark007r_results
    if r["arm"] == "HIGH"
})

ark007r_summary = {
    "n_pairs": n_pairs,
    "fresh_acquisition_seeds_completed": fresh_seeds_completed,
    "high_lr_collapses": high_c,
    "low_lr_collapses": low_c,
    "risk_difference_low_minus_high": risk_diff,
    "discordant_high_collapse_low_stable": discordant_protect,
    "discordant_reverse": discordant_reverse,
    "both_collapse": both_collapse,
    "both_stable": both_stable,
    "claim_level": (
        "FRESH_CHECKPOINT_REPLICATION_SUPPORTED"
        if len(fresh_seeds_completed) >= 2 and n_pairs >= 8 and discordant_protect > discordant_reverse
        else "INCOMPLETE_OR_INCONCLUSIVE"
    ),
}

save_result("ARK-007R_RESULT.json", {
    "experiment": "ARK-007R",
    "manifest_sha256": T2["sha"],
    "design": "fresh acquisition checkpoints + paired identical continuation orders",
    "summary": ark007r_summary,
    "results": ark007r_results,
})
print("\nARK-007R SUMMARY:", json.dumps(ark007r_summary, indent=2))


## ARK-009 — non-arithmetic transfer
Acquisition is a hard gate. If the task cannot be learned, retention is not tested or claimed.


In [ ]:
# ===== ARK-009: LEARNABLE NON-ARITHMETIC VARIABLE-BINDING TRANSFER =====
# Only proceed if enough budget remains. Acquisition is a hard gate.
if minutes_left() < 45:
    print("ARK-009 skipped: insufficient remaining budget", round(minutes_left(), 1), "min")
    ark009_payload = {"status": "SKIPPED_BUDGET"}
    save_result("ARK-009_RESULT.json", {"experiment": "ARK-009", **ark009_payload})
else:
    KEYS = list("ABCDEF")
    VALUES = list("uvwxyz")

    def build_binding_dataset(seed=4242, n_train=1200, n_test=300):
        rng = random.Random(seed)
        examples, seen_fact_sets = [], set()
        target = n_train + n_test
        i = 0
        # Split by complete fact-set identity, not merely prompt/query identity.
        # Therefore no test binding assignment occurs in train under another query.
        while len(examples) < target:
            i += 1
            ks = rng.sample(KEYS, 3)
            vs = rng.sample(VALUES, 3)
            rng.shuffle(vs)
            facts = list(zip(ks, vs))
            fact_sig = tuple(sorted(facts))
            if fact_sig in seen_fact_sets:
                continue
            seen_fact_sets.add(fact_sig)
            q_idx = (i + seed) % 3
            qk, ans = facts[q_idx]
            prompt = ";".join(f"{k}:{v}" for k, v in facts) + f";?{qk}="
            # deterministic query-swap to a different fact
            sk, sans = facts[(q_idx + 1) % 3]
            swap_prompt = ";".join(f"{k}:{v}" for k, v in facts) + f";?{sk}="
            examples.append({
                "prompt": prompt,
                "answer": ans,
                "swap_prompt": swap_prompt,
                "swap_answer": sans,
                "facts": facts,
                "query": qk,
                "fact_signature": fact_sig,
            })
        train_ex = examples[:n_train]
        test_ex = examples[n_train:]
        assert not (
            {tuple(e["fact_signature"]) for e in train_ex}
            & {tuple(e["fact_signature"]) for e in test_ex}
        )
        train = [(e["prompt"], e["answer"]) for e in train_ex]
        test = [(e["prompt"], e["answer"]) for e in test_ex]
        swap_test = [(e["swap_prompt"], e["swap_answer"]) for e in test_ex]
        payload = {"train": train, "test": test, "swap_test": swap_test}
        payload["sha"] = sha_json(payload)
        return payload

    BIND = build_binding_dataset()
    bind_vocab = CompactVocab()
    for split in ("train", "test", "swap_test"):
        for p, a in BIND[split]:
            bind_vocab.extend(p)
            bind_vocab.extend(a)

    print("ARK-009 task sha:", BIND["sha"], "| vocab", bind_vocab.size)

    ARK009_SEEDS = [1201, 1202]
    ARK009_CONT_SEEDS = [3701, 3702, 3703, 3704, 3705, 3706]
    ARK009_POST_STEPS = 6000
    ark009_acquired = {}
    ark009_results = []

    def acquire_binding(seed):
        torch.manual_seed(seed)
        model = Micro(bind_vocab.size, 128).to(DEVICE)
        opt = make_optimizer(model, 1e-3)
        rng = torch.Generator().manual_seed(seed)
        traj, steps, qual_values = [], [], []
        tokens = 0
        t0 = time.time()
        onset = confirm = None
        for step in range(1, 24001):
            idx = torch.randint(0, len(BIND["train"]), (64,), generator=rng)
            rows = [BIND["train"][i] for i in idx]
            loss, sup = loss_fn(model, bind_vocab, rows, DEVICE)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); xla_sync()
            tokens += sup

            if step % 200 == 0 or step == 1:
                te = greedy_exact(model, bind_vocab, BIND["test"], DEVICE, max_answer=4)
                swap = greedy_exact(model, bind_vocab, BIND["swap_test"], DEVICE, max_answer=4)
                score = min(te, swap / 0.85 * 0.90)  # only for a single sustained detector
                traj.append({"step": step, "test_exact": te, "query_swap_exact": swap})
                steps.append(step)
                qual_values.append(1.0 if (te >= 0.90 and swap >= 0.85) else 0.0)
                _, confirm = detect_sustained(steps, qual_values, bar=1.0, consec=3, below=False)
                if confirm is not None:
                    # onset of the qualifying 3-eval streak
                    onset = steps[-3]
                    return {
                        "status": "QUALIFIED",
                        "seed": seed,
                        "snapshot": snapshot_training_state(model, opt),
                        "reference_flat": flat_params(model).detach().cpu(),
                        "qualification_onset_step": onset,
                        "qualification_confirmation_step": confirm,
                        "acquisition_supervised_tokens": tokens,
                        "trajectory": traj,
                    }
            if time.time() - t0 > 1500 or minutes_left() < 25:
                break
        return {
            "status": "NO_QUALIFICATION",
            "seed": seed,
            "acquisition_supervised_tokens": tokens,
            "trajectory": traj,
        }

    def run_binding_continuation(acq, cs, arm_name, arm_lr):
        indices = generate_indices(cs, ARK009_POST_STEPS, 64, len(BIND["train"]))
        model, opt = restore_training_state(bind_vocab.size, acq["snapshot"], arm_lr)
        traj, tokens = [], 0
        for treated_step, batch_idx in enumerate(indices, start=1):
            rows = [BIND["train"][i] for i in batch_idx]
            loss, sup = loss_fn(model, bind_vocab, rows, DEVICE)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); xla_sync()
            tokens += sup
            if treated_step % 200 == 0:
                te = greedy_exact(model, bind_vocab, BIND["test"], DEVICE, max_answer=4)
                swap = greedy_exact(model, bind_vocab, BIND["swap_test"], DEVICE, max_answer=4)
                disp = param_displacement(model, acq["reference_flat"])
                traj.append({
                    "step": treated_step,
                    "test_exact": te,
                    "query_swap_exact": swap,
                    **disp,
                })
        return {
            "acquisition_seed": acq["seed"],
            "continuation_seed": cs,
            "continuation_order_sha256": order_hash(indices),
            "arm": arm_name,
            "lr": arm_lr,
            "qualification_onset_step": acq["qualification_onset_step"],
            "qualification_confirmation_step": acq["qualification_confirmation_step"],
            "retention": retention_metrics(traj),
            "final_query_swap_exact": traj[-1]["query_swap_exact"] if traj else None,
            "post_supervised_tokens": tokens,
            "trajectory": traj,
        }

    for seed in ARK009_SEEDS:
        if minutes_left() < 30:
            break
        print("\n=== ARK-009 acquire seed", seed, "===")
        acq = acquire_binding(seed)
        ark009_acquired[seed] = acq
        print("status:", acq["status"])
        if acq["status"] != "QUALIFIED":
            continue
        for cs in ARK009_CONT_SEEDS:
            if minutes_left() < 15:
                break
            for arm_name, arm_lr in (("HIGH", 1e-3), ("LOW", 1e-5)):
                print(f"  binding s{seed} order {cs} {arm_name}")
                ark009_results.append(run_binding_continuation(acq, cs, arm_name, arm_lr))

    qualified = sorted(seed for seed, a in ark009_acquired.items() if a["status"] == "QUALIFIED")
    if not qualified:
        transfer_status = "TRANSFER_BLOCKED_BY_ACQUISITION"
    else:
        bpairs = {}
        for r in ark009_results:
            bpairs.setdefault((r["acquisition_seed"], r["continuation_seed"]), {})[r["arm"]] = r
        complete_pairs = [p for p in bpairs.values() if "HIGH" in p and "LOW" in p]
        high_coll = sum(p["HIGH"]["retention"]["collapse_binary"] for p in complete_pairs)
        low_coll = sum(p["LOW"]["retention"]["collapse_binary"] for p in complete_pairs)
        transfer_status = (
            "TRANSFER_RETENTION_EFFECT_SUPPORTED"
            if len(qualified) >= 1 and len(complete_pairs) >= 4 and high_coll > low_coll
            else "TRANSFER_ACQUIRED_BUT_RETENTION_INCONCLUSIVE"
        )

    save_result("ARK-009_RESULT.json", {
        "experiment": "ARK-009",
        "task": "symbolic variable-binding retrieval",
        "task_sha256": BIND["sha"],
        "qualification": "3 consecutive evals: test>=0.90 and query-swap>=0.85",
        "qualified_seeds": qualified,
        "status": transfer_status,
        "acquisition": {
            str(k): {
                kk: vv for kk, vv in v.items()
                if kk not in ("snapshot", "reference_flat")
            }
            for k, v in ark009_acquired.items()
        },
        "results": ark009_results,
    })
    print("\nARK-009 STATUS:", transfer_status, "| qualified seeds:", qualified)


## ARK-010 — recovery after collapse
Uses only collapse events produced prospectively by ARK-007R in this same session.


In [ ]:
# ===== ARK-010: RECOVERY AFTER CONFIRMED COLLAPSE =====
if len(ark010_collapse_snapshots) < 2:
    ark010_status = "INCONCLUSIVE_LOW_EVENT_RATE"
    ark010_results = []
    print("ARK-010:", ark010_status, "| collapse snapshots:", len(ark010_collapse_snapshots))
else:
    ark010_results = []
    for event in sorted(
        ark010_collapse_snapshots,
        key=lambda e: (e["acquisition_seed"], e["continuation_seed"])
    ):
        if minutes_left() < 10:
            print("ARK-010 budget gate")
            break
        tail = event["tail_indices"]
        if len(tail) < 600:
            continue

        for arm_name, arm_lr in (("HIGH_CONTINUE", 1e-3), ("RECOVERY_LOW", 1e-5)):
            model, opt = restore_training_state(event["vocab_size"], event["snapshot"], arm_lr)
            traj = []
            vals, eval_steps = [], []
            rec_onset = rec_confirm = None

            for step, batch_idx in enumerate(tail, start=1):
                rows = [T2["train"][i] for i in batch_idx]
                loss, sup = loss_fn(model, CompactVocab(), rows, DEVICE)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); xla_sync()

                if step % 200 == 0:
                    te = greedy_exact(model, CompactVocab(), T2["test"], DEVICE)
                    disp = param_displacement(model, event["reference_flat"])
                    traj.append({"step": step, "test_exact": te, **disp})
                    eval_steps.append(step); vals.append(te)
                    rec_onset, rec_confirm = detect_sustained(
                        eval_steps, vals, bar=0.90, consec=3, below=False
                    )

            ark010_results.append({
                "source_acquisition_seed": event["acquisition_seed"],
                "source_continuation_seed": event["continuation_seed"],
                "collapse_onset_step": event["collapse_onset_step"],
                "collapse_confirmation_step": event["collapse_confirmation_step"],
                "arm": arm_name,
                "lr": arm_lr,
                "recovery90_onset": rec_onset,
                "recovery90_confirmation": rec_confirm,
                "recovered": rec_confirm is not None,
                "final_ood": traj[-1]["test_exact"] if traj else None,
                "trajectory": traj,
            })

    complete_events = {}
    for r in ark010_results:
        key = (r["source_acquisition_seed"], r["source_continuation_seed"])
        complete_events.setdefault(key, {})[r["arm"]] = r
    paired_events = [p for p in complete_events.values() if "HIGH_CONTINUE" in p and "RECOVERY_LOW" in p]
    low_rec = sum(p["RECOVERY_LOW"]["recovered"] for p in paired_events)
    high_rec = sum(p["HIGH_CONTINUE"]["recovered"] for p in paired_events)
    ark010_status = (
        "LOW_LR_RECOVERY_SUPPORTED"
        if len(paired_events) >= 2 and low_rec > high_rec
        else "RECOVERY_NOT_ESTABLISHED"
    )

save_result("ARK-010_RESULT.json", {
    "experiment": "ARK-010",
    "status": ark010_status,
    "source_collapse_events": len(ark010_collapse_snapshots),
    "results": ark010_results,
})
print("\nARK-010 STATUS:", ark010_status)


## Save receipts and download


In [ ]:
# ===== FINAL PROGRAM SUMMARY + DOWNLOAD =====
summary = {
    "plan_commit_sha": PLAN_SHA,
    "device": str(DEVICE),
    "torch": torch.__version__,
    "elapsed_minutes": (time.time() - SESSION_START) / 60.0,
    "minutes_left": minutes_left(),
    "files": sorted(p.name for p in RESULTS_DIR.glob("*.json")),
    "ARK-007R": ark007r_summary if "ark007r_summary" in globals() else {"status": "NOT_RUN"},
    "ARK-009": (
        {"status": transfer_status, "qualified_seeds": qualified}
        if "transfer_status" in globals()
        else {"status": "NOT_RUN"}
    ),
    "ARK-010": {"status": ark010_status} if "ark010_status" in globals() else {"status": "NOT_RUN"},
}
save_result("PROGRAM_SUMMARY_V3.json", summary)

zip_path = Path("/content/arkenstone_v3_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(RESULTS_DIR.glob("*")):
        if p.is_file():
            zf.write(p, arcname=p.name)

print("\n=== CAMPAIGN COMPLETE/PARTIAL ===")
print(json.dumps(summary, indent=2))
print("ZIP:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print("Auto-download unavailable:", exc)
    print("Download manually from:", zip_path)
